# 📦 Project Task: SiCepat Ekspres — First-Mile Logistics Analytics
## Data Cleaning, Feature Engineering & Exploratory Data Analysis

**Module 2 — Python for Data Analysis | Purwadhika Digital Technology School**

---

> ⚠️ **Jangan di-run dulu.** Copy notebook ini terlebih dahulu, baru kerjakan di file copy-an kamu.

---

### Konteks Bisnis
SiCepat Ekspres merebut hati jutaan seller UMKM melalui layanan First-Mile Pickup gratis dan paket murah HALU. Namun pertumbuhan masif ini memunculkan dua masalah kritis: **Phantom Pickup** oleh kurir yang manipulasi data untuk hindari denda KPI, dan **revenue leakage** miliaran rupiah akibat seller yang sengaja memperkecil berat paket di aplikasi.

Kamu berperan sebagai Data Analyst di tim **Business Intelligence SiCepat** yang diminta untuk membersihkan data operasional, mengidentifikasi pola Phantom Pickup dan manipulasi berat, serta memberikan rekomendasi berbasis data untuk SLA enforcement dan revenue recovery.

**Dataset (3 tabel):**
- `sicepat_sellers.csv` — 15.000 baris (Dimensi Seller)
- `sicepat_services.csv` — 5 baris (Dimensi Layanan)
- `sicepat_pickups.csv` — 300.000 baris (Fakta Pickup & Berat)

---

### ⚠️ Catatan Penting
- Task ini **open-ended** — tidak ada satu jawaban yang mutlak benar
- Yang dinilai: **ketepatan keputusan**, **kualitas justifikasi**, dan **kedalaman analisis**
- Setiap keputusan di Data Cleaning & Feature Engineering **wajib disertai penjelasan** di markdown cell
- EDA dikerjakan **tanpa visualisasi** — gunakan pandas aggregation, filtering, sorting, dan merge

---

### 🚨 Business Context Error (Wajib Diinvestigasi)
> **~10.199 transaksi** dengan `pickup_status = 'Success'` memiliki `pickup_time` yang terjadi **SEBELUM** `request_time`.  
> Ini adalah **Phantom Pickup**: kurir menekan tombol 'Pickup Selesai' sebelum seller bahkan memanggil.  
> Logika mustahil secara operasional — identifikasi, kuantifikasi, dan rekomendasikan mekanisme deteksi otomatis.

---
## 0. Import & Load Data

In [4]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [45]:
# Load semua dataset
# Sesuaikan path dengan lokasi file kamu
df_sellers_raw  = pd.read_csv('sicepat_sellers.csv')
df_services_raw = pd.read_csv('sicepat_services.csv')
df_pickups_raw  = pd.read_csv('sicepat_pickups.csv')

# Buat copy untuk dikerjakan
df_sellers  = df_sellers_raw.copy()
df_services = df_services_raw.copy()
df_pickups  = df_pickups_raw.copy()

print(f'sellers  : {df_sellers.shape}')
print(f'services : {df_services.shape}')
print(f'pickups  : {df_pickups.shape}')

sellers  : (15000, 4)
services : (5, 2)
pickups  : (300000, 9)


---
## 2. Data Cleaning

### 2.1 Eksplorasi Awal (Wajib)

Lakukan eksplorasi menyeluruh pada **ketiga tabel** sebelum membersihkan data apapun.

In [46]:
# Shape dan info umum — lakukan untuk ketiga tabel

df_sellers.info()
df_services.info()
df_pickups.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   seller_id    15000 non-null  str  
 1   seller_name  15000 non-null  str  
 2   city         15000 non-null  str  
 3   join_date    15000 non-null  str  
dtypes: str(4)
memory usage: 1.2 MB
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   service_code  5 non-null      str  
 1   service_name  5 non-null      str  
dtypes: str(2)
memory usage: 271.0 bytes
<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 9 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   resi_no                  300000 non-null  str    
 1   seller_id                300000 non-null  str    
 2   service_c

In [47]:
# Tipe data seluruh kolom
print(df_sellers.dtypes)
print(df_services.dtypes)
print(df_pickups.dtypes)

seller_id      str
seller_name    str
city           str
join_date      str
dtype: object
service_code    str
service_name    str
dtype: object
resi_no                        str
seller_id                      str
service_code                   str
item_category                  str
request_time                   str
pickup_time                    str
stated_weight_kg           float64
actual_volume_weight_kg    float64
pickup_status                  str
dtype: object


In [48]:
# Missing values: jumlah dan persentase per kolom, per tabel
# Tampilkan hanya kolom yang memiliki missing values
print("Missing Value Sellers:")
print(df_sellers.isnull().sum())

print("\nMissing Value Services:")
print(df_services.isnull().sum())

print("\nMissing Value Pickups:")
print(df_pickups.isnull().sum())

na_value3  = df_pickups.isnull().sum()
print("\nPersentase Missing Value Pickups:")
print((df_pickups.isnull().mean() * 100).round(2))
na_value3.apply(lambda  x:f"{x} - {x/len(df_pickups):.2%}")

Missing Value Sellers:
seller_id      0
seller_name    0
city           0
join_date      0
dtype: int64

Missing Value Services:
service_code    0
service_name    0
dtype: int64

Missing Value Pickups:
resi_no                        0
seller_id                      0
service_code                   0
item_category              30009
request_time                   0
pickup_time                22448
stated_weight_kg               0
actual_volume_weight_kg        0
pickup_status                  0
dtype: int64

Persentase Missing Value Pickups:
resi_no                    0.00
seller_id                  0.00
service_code               0.00
item_category             10.00
request_time               0.00
pickup_time                7.48
stated_weight_kg           0.00
actual_volume_weight_kg    0.00
pickup_status              0.00
dtype: float64


resi_no                         0 - 0.00%
seller_id                       0 - 0.00%
service_code                    0 - 0.00%
item_category              30009 - 10.00%
request_time                    0 - 0.00%
pickup_time                 22448 - 7.48%
stated_weight_kg                0 - 0.00%
actual_volume_weight_kg         0 - 0.00%
pickup_status                   0 - 0.00%
dtype: str

In [49]:
# Distribusi kolom-kolom kritis
# stated_weight_kg, actual_volume_weight_kg, pickup_status, item_category

# 1. Distribusi untuk kolom numerik (stated_weight_kg & actual_volume_weight_kg)
print("--- Statistik Deskriptif Kolom Numerik ---")
print(df_pickups[['stated_weight_kg', 'actual_volume_weight_kg']].describe())

print("\n--- Distribusi Kolom Kategori: pickup_status ---")
# Menampilkan frekuensi dan persentase pickup_status
status_dist = df_pickups['pickup_status'].value_counts(dropna=False)
status_pct = df_pickups['pickup_status'].value_counts(dropna=False, normalize=True) * 100
print(pd.DataFrame({'Jumlah': status_dist, 'Persentase (%)': status_pct}))

print("\n--- Distribusi Kolom Kategori: item_category ---")
# Menampilkan frekuensi dan persentase item_category (termasuk NaN)
cat_dist = df_pickups['item_category'].value_counts(dropna=False)
cat_pct = df_pickups['item_category'].value_counts(dropna=False, normalize=True) * 100
print(pd.DataFrame({'Jumlah': cat_dist, 'Persentase (%)': cat_pct}))

--- Statistik Deskriptif Kolom Numerik ---
       stated_weight_kg  actual_volume_weight_kg
count         300000.00                300000.00
mean               6.15                     4.15
std               58.34                     2.84
min               -1.50                     0.50
25%                1.61                     2.17
50%                2.75                     3.55
75%                3.88                     4.88
max              999.90                    14.99

--- Distribusi Kolom Kategori: pickup_status ---
               Jumlah  Persentase (%)
pickup_status                        
Success        254992           85.00
Failed          30090           10.03
Rescheduled     14918            4.97

--- Distribusi Kolom Kategori: item_category ---
               Jumlah  Persentase (%)
item_category                        
Baju            45250           15.08
Kosmetik        44985           14.99
baju            30040           10.01
NaN             30009           10.0

In [50]:
# Cek konsistensi relasi antar tabel

# 1. Cek apakah semua service_code di pickups ada di tabel services
check_services = df_pickups['service_code'].isin(df_services['service_code']).all()
print(f"Apakah semua service_code di pickups ada di services? {check_services}")

# 2. Cek apakah semua seller_id di pickups ada di tabel sellers
check_sellers = df_pickups['seller_id'].isin(df_sellers['seller_id']).all()
print(f"Apakah semua seller_id di pickups ada di sellers? {check_sellers}")

Apakah semua service_code di pickups ada di services? True
Apakah semua seller_id di pickups ada di sellers? True


**✍️ Ringkasan Temuan Eksplorasi:**

*(Kolom apa yang bermasalah di setiap tabel, seberapa parah, dan prioritas penanganan kamu)*

Untuk Tabel sicepat_pickups kolom pickup_time banyak yang mendahului request_time. Ini parah karena berarti kurir melakukan phantom pick up, penganannya nanti difilter menggunakan fungsi request time harus < Pick up time.

Kolom item_category pada tabel pickup banyak input yang tidak di kategorikan, seller mengisi sendiri. Jadi banyak yang di persingkat atau bahkan tidak diisi

Kolom Stated weight ada yang negatif

> 

---
### 2.2 Kerangka Identifikasi Missing Values

Sebelum menangani missing values pada kolom manapun, identifikasi dulu **jenis missing value-nya**.

| Jenis | Definisi Singkat | Implikasi Penanganan | Contoh di Dataset Ini |
|---|---|---|---|
| **MCAR** *(Missing Completely At Random)* | Nilai kosong tidak berkaitan dengan kolom lain. Pola missing benar-benar acak. | Relatif aman di-impute atau di-drop tanpa bias signifikan. | `item_category` kosong secara acak tanpa pola tertentu. |
| **MAR** *(Missing At Random)* | Nilai kosong berkaitan dengan kolom **lain**, bukan dengan nilai kolom itu sendiri. | Imputation berbasis kolom lain lebih tepat. Drop bisa menyebabkan bias. | `item_category` kosong lebih sering pada seller tertentu yang malas mengisi formulir. |
| **MNAR** *(Missing Not At Random)* | Nilai kosong berkaitan langsung dengan nilai yang seharusnya ada. Ada alasan sistematis. | Imputation apapun berisiko misleading. Perlu keputusan bisnis eksplisit. | `pickup_time` kosong karena transaksi Failed/Rescheduled — nilai kosong itu sendiri adalah informasi operasional. |

> 💡 Justifikasi reasoning kamu lebih penting dari labelnya.

---
### 2.3 Penanganan `item_category`

Kolom ini diisi manual oleh seller di aplikasi e-commerce, menghasilkan **11 varian penulisan** untuk ~6 kategori, ditambah ~10% missing (~30.009 baris).

| Varian Asli | Kategori Standar yang Dimaksud |
|---|---|
| `Baju`, `baju`, `Pakaian`, `Fashion` | Fashion & Pakaian |
| `Kosmetik`, `ksmtk` | Kecantikan |
| `Skincare` | Kecantikan (atau terpisah?) |
| `Elektronik`, `hp` | Elektronik |
| `Sepatu`, `spt` | Alas Kaki |
| `NaN` (~30.009 baris) | Unknown / Perlu keputusan |

> 🧠 **Critical Thinking Prompt:**  
> Apakah 'Kosmetik' dan 'Skincare' benar-benar sama? Keduanya mungkin memiliki profil berat volumetrik berbeda.  
> Pertimbangkan kebutuhan analisis downstream sebelum memutuskan untuk menggabungkan atau memisahkan.

In [53]:
# Lihat semua nilai unik item_category beserta frekuensinya
df_pickups["item_category"].value_counts(dropna=False)

category_summary = pd.DataFrame({
    'Jumlah': df_pickups['item_category'].value_counts(dropna=False),
    'Persentase (%)': df_pickups['item_category'].value_counts(dropna=False, normalize=True) * 100
})
category_summary

,Jumlah,Persentase (%)
item_category,,
Baju,45250,15.08
Kosmetik,44985,14.99
baju,30040,10.01
NaN,30009,10.00
Fashion,29942,9.98
Pakaian,29874,9.96
Skincare,29802,9.93
Elektronik,15087,5.03
ksmtk,15003,5.00


In [54]:
avg_actual_weight =df_pickups.groupby('item_category', dropna=False)['actual_volume_weight_kg'].agg(
    Rata_Rata_kg='mean',
    Median_kg='median',
    Jumlah_Transaksi='count'
).reset_index()

avg_actual_weight

,item_category,Rata_Rata_kg,Median_kg,Jumlah_Transaksi
0,Baju,4.16,3.55,45250
1,Elektronik,4.13,3.54,15087
2,Fashion,4.17,3.57,29942
3,Kosmetik,4.16,3.55,44985
4,Pakaian,4.15,3.54,29874
5,Sepatu,4.10,3.53,12025
6,Skincare,4.16,3.56,29802
7,baju,4.16,3.56,30040
8,hp,4.17,3.57,14975
9,ksmtk,4.16,3.56,15003


**✍️ Mapping standarisasi yang kamu buat:**
- Varian asli → nilai standar (tuliskan mapping lengkapnya):
- Apakah 'Skincare' digabung dengan 'Kosmetik' atau dipisah? Alasan:
- **Jenis missing value (MCAR / MAR / MNAR):** dan alasan klasifikasi kamu:
- Keputusan penanganan missing values dan alasan:

> 

In [43]:
# TODO: Standarisasi item_category
# Simpan hasil ke kolom baru: item_category_clean
# Mapping dictionary untuk standarisasi item_category
category_mapping = {
    'Baju': 'Fashion & Pakaian',
    'baju': 'Fashion & Pakaian',
    'Pakaian': 'Fashion & Pakaian',
    'Fashion': 'Fashion & Pakaian',
    'Kosmetik': 'Kecantikan & Skincare',
    'ksmtk': 'Kecantikan & Skincare',
    'Skincare': 'Kecantikan & Skincare',
    'Elektronik': 'Elektronik',
    'hp': 'Elektronik',
    'Sepatu': 'Alas Kaki',
    'spt': 'Alas Kaki'
}

# Standarisasi dan simpan ke kolom baru item_category_clean
df_pickups['item_category_clean'] = df_pickups['item_category'].map(category_mapping).fillna('Unknown')

In [44]:
df_pickups

,resi_no,seller_id,service_code,item_category,request_time,pickup_time,stated_weight_kg,actual_volume_weight_kg,pickup_status,item_category_clean,weight_gap_kg,is_weight_manipulated
0,000SC000000001,SEL-04246,SVC-05,ksmtk,2023-08-20 15:00:00,2023-08-20 20:11:58.720564,2.51,2.98,Success,Kecantikan,0.47,0
1,000SC000000002,SEL-04881,SVC-02,Pakaian,2023-08-14 06:00:00,2023-08-15 01:43:18.472104,1.50,11.39,Success,Fashion & Pakaian,9.89,0
2,000SC000000003,SEL-10176,SVC-02,Skincare,2023-09-04 09:00:00,2023-09-05 01:54:57.252587,4.00,4.04,Success,Kecantikan,0.04,0
3,000SC000000004,SEL-05400,SVC-02,Kosmetik,2023-11-10 13:00:00,2023-11-11 04:30:11.251160,4.74,5.16,Rescheduled,Kecantikan,0.42,0
4,000SC000000005,SEL-08010,SVC-02,NaN,2023-10-03 08:00:00,2023-10-04 07:26:21.201376,4.40,11.69,Success,Unknown,7.29,0
...,...,...,...,...,...,...,...,...,...,...,...,...
299995,000SC000299996,SEL-04921,SVC-02,Kosmetik,2023-10-25 07:00:00,2023-10-25 21:59:07.346574,3.86,12.19,Success,Kecantikan,8.33,0
299996,000SC000299997,SEL-13537,SVC-03,Pakaian,2023-10-12 19:00:00,2023-10-13 18:15:18.236227,2.12,9.56,Success,Fashion & Pakaian,7.44,0
299997,000SC000299998,SEL-04746,SVC-03,Baju,2023-10-03 16:00:00,2023-10-04 04:02:48.523948,4.30,4.57,Success,Fashion & Pakaian,0.27,0
299998,000SC000299999,SEL-01880,SVC-05,Kosmetik,2023-08-09 06:00:00,2023-08-09 22:01:10.188472,0.67,5.68,Success,Kecantikan,5.01,0


---
### 2.4 Penanganan `stated_weight_kg`

Kolom ini memiliki tiga jenis anomali berbeda yang masing-masing butuh penanganan terpisah:

| Tipe Anomali | Jumlah Baris (approx.) | Kemungkinan Penyebab |
|---|---|---|
| Nilai negatif (< 0) | ~969 baris | Input error seller, bug validasi form |
| Nilai sangat kecil (< 0.1 kg) | ~1.970 baris | Default value yang tidak diubah, atau manipulasi disengaja |
| Nilai sangat besar (> 100 kg) | ~1.030 baris | Salah satuan (gram vs kg), atau paket industri |

> 🧠 **Critical Thinking Prompt:**  
> Nilai stated_weight yang sangat kecil (0.01 kg) mungkin adalah seller yang **sengaja mengecilkan berat** untuk hemat ongkir.  
> Ini adalah potensi revenue leakage yang berbeda dari sekadar input error.  
> Apakah keputusan kamu berbeda jika anomali tersebut berkorelasi dengan `weight_gap` yang besar?

In [94]:
# Investigasi distribusi stated_weight_kg secara menyeluruh
# Identifikasi ketiga jenis anomali dan jumlah baris terdampak

# 1. Investigasi distribusi dan identifikasi anomali
negatif = df_pickups[df_pickups['stated_weight_kg'] < 0]
sangat_kecil = df_pickups[(df_pickups['stated_weight_kg'] >= 0) & (df_pickups['stated_weight_kg'] < 0.1)]
sangat_besar = df_pickups[df_pickups['stated_weight_kg'] > 100]

print(f"Jumlah Nilai Negatif: {len(negatif)}")
print(f"Jumlah Nilai Sangat Kecil (< 0.1 kg): {len(sangat_kecil)}")
print(f"Jumlah Nilai Sangat Besar (> 100 kg): {len(sangat_besar)}")

Jumlah Nilai Negatif: 0
Jumlah Nilai Sangat Kecil (< 0.1 kg): 0
Jumlah Nilai Sangat Besar (> 100 kg): 0


In [91]:
# Investigasi lanjutan: apakah anomali berkorelasi dengan seller, service, atau item_category tertentu?
# 1. Menandai baris yang merupakan anomali pada stated_weight_kg
# (Misal: nilai negatif, < 0.1 kg, atau > 100 kg)
# Buat indikator anomali sementara
df_pickups['is_anomaly'] = (
    (df_pickups['stated_weight_kg'] < 0)
    | (df_pickups['stated_weight_kg'] < 0.1)
    | (df_pickups['stated_weight_kg'] > 100)
)

# Investigasi berdasarkan Seller
seller_corr = (
    df_pickups.groupby('seller_id')['is_anomaly']
    .agg(total_orders='count', anomaly_count='sum', anomaly_rate='mean')
    .sort_values(by='anomaly_count', ascending=False)
)
print('--- Korelasi Anomali per Seller (Top 10) ---')
print(seller_corr[seller_corr['total_orders'] >= 5].head(10))

# Investigasi berdasarkan Service Code
service_col = (
    'service_code' if 'service_code' in df_pickups.columns else 'service_type'
)
if service_col in df_pickups.columns:
    service_corr = df_pickups.groupby(service_col)['is_anomaly'].agg(
        total_orders='count', anomaly_count='sum', anomaly_rate='mean'
    )
    print(f'\n--- Korelasi Anomali per {service_col} ---')
    print(service_corr)

# Investigasi berdasarkan Item Category
if 'item_category' in df_pickups.columns:
    category_corr = df_pickups.groupby('item_category')['is_anomaly'].agg(
        total_orders='count', anomaly_count='sum', anomaly_rate='mean'
    )
    print('\n--- Korelasi Anomali per Item Category ---')
    print(category_corr)

--- Korelasi Anomali per Seller (Top 10) ---
           total_orders  anomaly_count  anomaly_rate
seller_id                                           
SEL-00001            14              0          0.00
SEL-00002            24              0          0.00
SEL-00003            25              0          0.00
SEL-00004            18              0          0.00
SEL-00005            20              0          0.00
SEL-00006            16              0          0.00
SEL-00007            17              0          0.00
SEL-00008            25              0          0.00
SEL-00009            21              0          0.00
SEL-00010            16              0          0.00

--- Korelasi Anomali per service_code ---
              total_orders  anomaly_count  anomaly_rate
service_code                                           
SVC-01              150107              0          0.00
SVC-02               45079              0          0.00
SVC-03               29874              0          0

**✍️ Analisis & Justifikasi per tipe anomali:**
- **Nilai negatif** — hipotesis penyebab, keputusan penanganan, alasan:
- **Nilai sangat kecil (< 0.1 kg)** — apakah ini error atau manipulasi disengaja? Keputusan:
- **Nilai sangat besar (> 100 kg)** — threshold 'wajar' yang kamu pilih dan justifikasinya:
- Apakah keputusan kamu berbeda jika nilai anomali berkorelasi dengan weight_gap besar?

> 

In [ ]:
# TODO: Implementasi penanganan anomali stated_weight_kg
df_pickups['stated_weight_clean'] = df_pickups['stated_weight_kg']
df_pickups['suspicious_weight'] = False

# A. Penanganan Nilai Negatif -> Absolutkan nilai
df_pickups['stated_weight_clean'] = df_pickups['stated_weight_clean'].abs()

# B. Penanganan Nilai Sangat Kecil (< 0.1 kg) -> Flagging & Imputasi
small_weight_mask = df_pickups['stated_weight_clean'] < 0.1
df_pickups.loc[small_weight_mask, 'suspicious_weight'] = True

# Imputasi nilai sangat kecil dengan actual_weight_kg atau median kategori
if 'actual_weight_kg' in df_pickups.columns:
    df_pickups.loc[small_weight_mask, 'stated_weight_clean'] = df_pickups.loc[
        small_weight_mask, 'actual_weight_kg'
    ]

# C. Penanganan Nilai Sangat Besar (> 100 kg) -> Koreksi Gram ke Kg / Caping
large_weight_mask = df_pickups['stated_weight_clean'] > 100
df_pickups.loc[large_weight_mask, 'suspicious_weight'] = True


# Koreksi kesalahan input gram (asumsi nilai 100 kg - 1000 kg adalah gram)
def fix_large_weight(val):
    if pd.isna(val):
        return val
    if val > 1000:
        return 50.0  # Capping ke batas wajar 50 kg untuk e-commerce UMKM
    if val > 100:
        return val / 1000.0  # Konversi gram ke kg
    return val


df_pickups['stated_weight_clean'] = df_pickups['stated_weight_clean'].apply(
    fix_large_weight
)

# Imputasi sisa nilai NaN (jika ada) menggunakan median per kategori
if 'item_category' in df_pickups.columns:
    df_pickups['stated_weight_clean'] = df_pickups.groupby('item_category')[
        'stated_weight_clean'
    ].transform(lambda x: x.fillna(x.median()))

df_pickups['stated_weight_clean'] = df_pickups['stated_weight_clean'].fillna(
    df_pickups['stated_weight_clean'].median()
)

# Hapus kolom temporer
df_pickups.drop(columns=['is_anomaly'], inplace=True, errors='ignore')



Penanganan anomali stated_weight_kg selesai diimplementasikan.


In [93]:
df_pickups

,resi_no,seller_id,service_code,item_category,request_time,pickup_time,stated_weight_kg,actual_volume_weight_kg,pickup_status,weight_gap_kg,is_weight_manipulated,item_category_clean,stated_weight_clean,suspicious_weight
0,000SC000000001,SEL-04246,SVC-05,ksmtk,2023-08-20 15:00:00,2023-08-20 20:11:58.720564,2.51,2.98,Success,0.47,0,Kecantikan,2.51,False
1,000SC000000002,SEL-04881,SVC-02,Pakaian,2023-08-14 06:00:00,2023-08-15 01:43:18.472104,1.50,11.39,Success,9.89,0,Fashion & Pakaian,1.50,False
2,000SC000000003,SEL-10176,SVC-02,Skincare,2023-09-04 09:00:00,2023-09-05 01:54:57.252587,4.00,4.04,Success,0.04,0,Kecantikan,4.00,False
3,000SC000000004,SEL-05400,SVC-02,Kosmetik,2023-11-10 13:00:00,2023-11-11 04:30:11.251160,4.74,5.16,Rescheduled,0.42,0,Kecantikan,4.74,False
4,000SC000000005,SEL-08010,SVC-02,NaN,2023-10-03 08:00:00,2023-10-04 07:26:21.201376,4.40,11.69,Success,7.29,0,Unknown,2.74,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299995,000SC000299996,SEL-04921,SVC-02,Kosmetik,2023-10-25 07:00:00,2023-10-25 21:59:07.346574,3.86,12.19,Success,8.33,0,Kecantikan,3.86,False
299996,000SC000299997,SEL-13537,SVC-03,Pakaian,2023-10-12 19:00:00,2023-10-13 18:15:18.236227,2.12,9.56,Success,7.44,0,Fashion & Pakaian,2.12,False
299997,000SC000299998,SEL-04746,SVC-03,Baju,2023-10-03 16:00:00,2023-10-04 04:02:48.523948,4.30,4.57,Success,0.27,0,Fashion & Pakaian,4.30,False
299998,000SC000299999,SEL-01880,SVC-05,Kosmetik,2023-08-09 06:00:00,2023-08-09 22:01:10.188472,0.67,5.68,Success,5.01,0,Kecantikan,0.67,False


---
### 2.5 Penanganan `pickup_time` yang Kosong

Kolom `pickup_time` memiliki **~22.448 nilai kosong** (~7.5% dari total). Sebelum diisi atau di-drop, investigasi dulu pola missing-nya.

> 🧠 **Critical Thinking Prompt:**  
> `pickup_time` yang kosong pada transaksi Failed adalah **informasi bisnis**, bukan data rusak.  
> Mengisi NaN dengan nilai apapun akan merusak integritas analisis SLA dan Phantom Pickup.  
> Pertahankan NaN, dan pastikan analisis SLA hanya dilakukan pada baris dengan `pickup_time` yang valid.

In [95]:
# Investigasi: apakah missing pickup_time seluruhnya pada transaksi Failed dan Rescheduled?
# Breakdown missing pickup_time per pickup_status
# 1 & 2. Breakdown jumlah missing pickup_time berdasarkan pickup_status
missing_pickup = df_pickups[df_pickups["pickup_time"].isna()]

status_breakdown = (
    missing_pickup["pickup_status"]
    .value_counts(dropna=False)
    .reset_index(name="missing_count")
)

# Hitung persentase terhadap total missing
status_breakdown["percentage"] = (
    status_breakdown["missing_count"] / len(missing_pickup)
) * 100

print("=== Breakdown Missing pickup_time per pickup_status ===")
print(status_breakdown)

# Cek apakah ADA pickup_status 'Completed' / 'Success' yang pickup_time-nya NaN
is_all_failed_or_rescheduled = missing_pickup["pickup_status"].isin(
    ["Failed", "Rescheduled"]
).all()
print(
    f"\nApakah semua missing pickup_time hanya pada status Failed/Rescheduled? {is_all_failed_or_rescheduled}"
)

=== Breakdown Missing pickup_time per pickup_status ===
  pickup_status  missing_count  percentage
0        Failed          14973       66.70
1   Rescheduled           7475       33.30

Apakah semua missing pickup_time hanya pada status Failed/Rescheduled? True


**✍️ Analisis & Justifikasi:**
- **Jenis missing value (MCAR / MAR / MNAR):** dan alasan klasifikasi kamu:
- Temuan investigasi (apakah 100% missing pada Failed/Rescheduled?):
- Keputusan penanganan (pertahankan NaN / isi placeholder) dan alasan:
- Implikasi keputusan ini terhadap analisis SLA di Section 3 dan 4:

> 

In [96]:
# Konversi kolom datetime untuk kedua tabel yang relevan
# Lakukan di sini agar tersedia untuk Section 2.6
# Konversi Datetime
df_pickups["request_time"] = pd.to_datetime(
    df_pickups["request_time"]
)

df_pickups["pickup_time"] = pd.to_datetime(
    df_pickups["pickup_time"]
)

---
### 2.6 Penanganan Business Logic Error: Phantom Pickup

**Ini adalah anomali paling kritis di dataset ini.** Sebanyak ~10.199 transaksi dengan `pickup_status = 'Success'` memiliki `pickup_time` **SEBELUM** `request_time` — secara logika operasional mustahil: kurir tidak mungkin menjemput sebelum seller memanggil.

| Tipe Phantom Pickup | Deskripsi | Indikasi |
|---|---|---|
| **Phantom Ringan** | Selisih < 1 jam | Kemungkinan clock skew antar server, bukan manipulasi |
| **Phantom Berat** | Selisih ≥ 1 jam (rata-rata ~3 jam) | Indikasi kuat manipulasi data oleh kurir |

> 🚨 **Critical Thinking Prompt:**  
> **JANGAN drop baris Phantom Pickup** — ini adalah bukti operasional yang sangat berharga.  
> Data ini adalah dasar untuk membangun sistem deteksi kurir nakal dan program suspend otomatis.  
> **Flag, pertahankan, dan analisis secara terpisah.**

In [97]:
# Identifikasi Phantom Pickup: Success dengan pickup_time < request_time
# Berapa jumlahnya? Berapa persentase dari seluruh Success?
# Identifikasi Phantom Pickup: Success dengan pickup_time < request_time
# Berapa jumlahnya? Berapa persentase dari seluruh Success?
success_data = df_pickups[
    df_pickups["pickup_status"] == "Success"
]

phantom_data = success_data[
    success_data["pickup_time"] < success_data["request_time"]
]

print("Jumlah Phantom Pickup:")
print(len(phantom_data))

print("\nJumlah Success:")
print(len(success_data))

print("\nPersentase Phantom Pickup:")
print(
    len(phantom_data) /
    len(success_data) *
    100
)

Jumlah Phantom Pickup:
10199

Jumlah Success:
254992

Persentase Phantom Pickup:
3.9997333249670577


In [98]:
# Investigasi distribusi selisih waktu (jam) pada Phantom Pickup
# Hitung: (request_time - pickup_time) dalam jam untuk kasus Phantom
# Investigasi distribusi selisih waktu (jam) pada Phantom Pickup
# Hitung: (request_time - pickup_time) dalam jam untuk kasus Phantom
phantom_data = phantom_data.copy()

phantom_data["phantom_gap_hours"] = (
    phantom_data["request_time"] -
    phantom_data["pickup_time"]
).dt.total_seconds() / 3600

print(phantom_data["phantom_gap_hours"].describe())


count   10199.00
mean        2.99
std         1.13
min         1.17
25%         2.17
50%         3.17
75%         3.82
max         4.82
Name: phantom_gap_hours, dtype: float64


In [ ]:
# TODO: Buat flag is_phantom_pickup dan simpan ke df_pickups
# Pertimbangkan: apakah perlu flag terpisah untuk Phantom Ringan vs Phantom Berat?

print(
    "Phantom ringan:",
    (phantom_data["phantom_gap_hours"] < 1).sum()
)

print(
    "Phantom berat:",
    (phantom_data["phantom_gap_hours"] >= 1).sum()
)

Phantom ringan: 0
Phantom berat: 10199


In [103]:
# Flag Phantom
df_pickups["is_phantom_pickup"] = (
    (df_pickups["pickup_status"] == "Success") &
    (df_pickups["pickup_time"].notna()) &
    (df_pickups["pickup_time"] < df_pickups["request_time"])
)

In [104]:
# Investigasi lanjutan: apakah Phantom Pickup terkonsentrasi pada seller, kota, atau service tertentu?
# Investigasi lanjutan: apakah Phantom Pickup terkonsentrasi pada seller, kota, atau service tertentu?
# Flag Phantom


# Investigasi Phantom Berdasarkan Kota
phantom_city = df_pickups.merge(
    df_sellers[["seller_id", "city"]],
    on="seller_id",
    how="left"
)

phantom_city = phantom_city.groupby("city").agg(
    total_pickup=("resi_no", "count"),
    phantom_pickup=("is_phantom_pickup", "sum")
)

phantom_city["phantom_rate_percent"] = (
    phantom_city["phantom_pickup"] /
    phantom_city["total_pickup"] *
    100
)

phantom_city.sort_values(
    "phantom_rate_percent",
    ascending=False
)

,total_pickup,phantom_pickup,phantom_rate_percent
city,,,
Jakarta Pusat,33149,1159,3.50
Sidoarjo,33166,1149,3.46
Cimahi,34274,1182,3.45
Medan,34115,1173,3.44
Jakarta Selatan,33179,1140,3.44
Makassar,32258,1096,3.40
Surabaya,33119,1103,3.33
Bandung,32928,1095,3.33
Yogyakarta,33812,1102,3.26


In [105]:

# Investigasi Phantom Berdasarkan Seller
phantom_seller = df_pickups.groupby("seller_id").agg(
    total_pickup=("resi_no", "count"),
    phantom_pickup=("is_phantom_pickup", "sum")
)

phantom_seller["phantom_rate_percent"] = (
    phantom_seller["phantom_pickup"] /
    phantom_seller["total_pickup"] *
    100
)

phantom_seller[
    phantom_seller["total_pickup"] >= 20
].sort_values(
    "phantom_rate_percent",
    ascending=False
).head(10)

,total_pickup,phantom_pickup,phantom_rate_percent
seller_id,,,
SEL-00776,23,5,21.74
SEL-02557,24,5,20.83
SEL-08496,20,4,20.00
SEL-05873,20,4,20.00
SEL-09484,21,4,19.05
SEL-09512,21,4,19.05
SEL-07049,21,4,19.05
SEL-00649,21,4,19.05
SEL-10817,21,4,19.05


**✍️ Analisis & Justifikasi:**
- Jumlah dan persentase Phantom Pickup dari seluruh transaksi Success:
- Distribusi selisih waktu — apakah lebih banyak Phantom Ringan atau Phantom Berat?
- Pola konsentrasi yang ditemukan (kota/seller/service tertentu?):
- Keputusan penanganan — kolom flag apa yang kamu buat? Apakah kamu bedakan Phantom Ringan vs Berat?
- Hipotesis mengapa bug ini bisa terjadi di aplikasi kurir SiGESIT:

> 

In [ ]:
# TODO: Buat flag is_phantom_pickup dan simpan ke df_pickups
# Pertimbangkan: apakah perlu flag terpisah untuk Phantom Ringan vs Phantom Berat? 

---
### 2.7 Penanganan Duplikat & Integritas Data

In [106]:
# 1. Cek exact duplicates di setiap tabel
print("=== 1. Exact Duplicates ===")
print("df_pickups:", df_pickups.duplicated().sum())
print("df_sellers:", df_sellers.duplicated().sum())
print("df_services:", df_services.duplicated().sum())

=== 1. Exact Duplicates ===
df_pickups: 0
df_sellers: 0
df_services: 0


In [107]:
# 2. Cek duplikat resi_no
print("\n=== 2. Duplikat resi_no ===")
resi_dup_count = df_pickups.duplicated(subset=['resi_no'], keep=False).sum()
print(f"Jumlah baris dengan resi_no terduplikasi: {resi_dup_count}")


=== 2. Duplikat resi_no ===
Jumlah baris dengan resi_no terduplikasi: 0


In [108]:
# 3. Cek service_code di pickups yang tidak ada di services
print("\n=== 3. Orphan service_code ===")
missing_services = df_pickups[
    ~df_pickups['service_code'].isin(df_services['service_code'])
]
print(
    f"Jumlah baris pickup dengan service_code tidak terdaftar: {len(missing_services)}"
)


=== 3. Orphan service_code ===
Jumlah baris pickup dengan service_code tidak terdaftar: 0


In [109]:
# 4. Cek seller_id di pickups yang tidak ada di sellers
print("\n=== 4. Orphan seller_id ===")
missing_sellers = df_pickups[
    ~df_pickups['seller_id'].isin(df_sellers['seller_id'])
]
print(
    f"Jumlah baris pickup dengan seller_id tidak terdaftar: {len(missing_sellers)}"
)


=== 4. Orphan seller_id ===
Jumlah baris pickup dengan seller_id tidak terdaftar: 0


**✍️ Analisis & Justifikasi:**
- Masalah yang ditemukan dan jumlah baris terdampak:
- Hipotesis untuk setiap masalah:
- Keputusan penanganan per masalah:

> 

In [110]:
# TODO: Implementasi keputusan penanganan masalah integritas
# 1. Hapus exact duplicates pada seluruh DataFrame
df_pickups = df_pickups.drop_duplicates()
df_sellers = df_sellers.drop_duplicates()
df_services = df_services.drop_duplicates()

# 2. Hapus duplikat berdasarkan resi_no (pertahankan baris pertama)
df_pickups = df_pickups.drop_duplicates(subset=['resi_no'], keep='first')

# 3. Handling orphan service_code (Ganti dengan 'UNKNOWN' jika tidak ada di df_services)
valid_services = df_services['service_code'].unique()
df_pickups['service_code'] = df_pickups['service_code'].apply(
    lambda x: x if x in valid_services else 'UNKNOWN'
)

# 4. Handling orphan seller_id (Ganti dengan 'UNKNOWN_SELLER' jika tidak ada di df_sellers)
valid_sellers = df_sellers['seller_id'].unique()
df_pickups['seller_id'] = df_pickups['seller_id'].apply(
    lambda x: x if x in valid_sellers else 'UNKNOWN_SELLER'
)


---
## 3. Feature Engineering

### 3.1 Fitur Wajib

Buat 8 kolom berikut. Sertakan penjelasan singkat business value-nya di setiap fitur.

#### ⚙️ `item_category_clean`
*(Sudah dibuat di Section 2.3 — pastikan sudah ada di df_pickups)*

#### ⚙️ `weight_gap_kg`

> 💡 `actual_volume_weight_kg − stated_weight_kg`. Nilai positif = seller *underdeclare* berat.  
> Gunakan `stated_weight_kg` yang **sudah di-clean** dari Section 2.4.

In [111]:
# TODO: Buat weight_gap_kg
# Business value: mengukur selisih berat yang menjadi dasar estimasi revenue leakage
df_pickups["weight_gap_kg"] = (
    df_pickups["actual_volume_weight_kg"] -
    df_pickups["stated_weight_kg"]
)

#### ⚙️ `is_oversize`

> 💡 Tentukan threshold kamu sendiri untuk mendefinisikan 'oversize'. Justifikasikan berdasarkan distribusi `weight_gap_kg`.

**✍️ Threshold yang kamu pilih dan alasannya:**

> 

In [112]:
# TODO: Buat is_oversize (boolean)
# Lihat distribusi weight_gap_kg terlebih dahulu untuk menentukan threshold
df_pickups["is_oversize"] = pd.Series(
    pd.NA,
    index=df_pickups.index,
    dtype="boolean"
)

valid_weight = df_pickups["stated_weight_kg"].notna()

df_pickups.loc[valid_weight, "is_oversize"] = (
    df_pickups.loc[valid_weight, "actual_volume_weight_kg"] >
    df_pickups.loc[valid_weight, "stated_weight_kg"] * 1.5
)

#### ⚙️ `is_phantom_pickup`
*(Sudah dibuat di Section 2.6 — pastikan sudah ada di df_pickups)*

#### ⚙️ `sla_hours`

> 💡 Hanya dihitung untuk transaksi `Success` yang **bukan** Phantom Pickup.  
> Untuk Failed, Rescheduled, dan Phantom Pickup: isi dengan NaN.

In [113]:
# TODO: Buat sla_hours
# Business value: mengukur durasi pickup dalam jam untuk analisis SLA compliance


# Pastikan kolom waktu berformat datetime
df_pickups['request_time'] = pd.to_datetime(df_pickups['request_time'])
df_pickups['pickup_time'] = pd.to_datetime(df_pickups['pickup_time'])

# Kondisi: Transaksi Success DAN bukan Phantom Pickup
mask_success = (df_pickups['pickup_status'] == 'Success') & (
    df_pickups['is_phantom_pickup'] == False
)

# Hitung durasi dalam jam
df_pickups['sla_hours'] = np.nan
df_pickups.loc[mask_success, 'sla_hours'] = (
    df_pickups.loc[mask_success, 'pickup_time']
    - df_pickups.loc[mask_success, 'request_time']
).dt.total_seconds() / 3600.0

#### ⚙️ `is_sla_met`

> 💡 SLA SiCepat First-Mile: 1x24 jam (≤ 24 jam). Null untuk transaksi non-Success.

In [ ]:
# TODO: Buat is_sla_met (boolean: True jika sla_hours <= 24)
# Business value: indikator utama performa kurir SiGESIT
# True jika sla_hours <= 24, False jika > 24, dan NaN untuk sisanya
df_pickups['is_sla_met'] = np.nan
df_pickups.loc[df_pickups['sla_hours'].notna(), 'is_sla_met'] = (
    df_pickups['sla_hours'] <= 24

#### ⚙️ `seller_tenure_days`

> 💡 Tentukan sendiri tanggal referensi yang kamu gunakan dan justifikasikan.

**✍️ Tanggal referensi yang kamu gunakan dan alasannya:**

> 

In [ ]:
# TODO: Buat seller_tenure_days di df_sellers
# Business value: indikator maturitas dan rekam jejak seller
# Pastikan kolom datetime
df_sellers['created_at'] = pd.to_datetime(df_sellers['created_at'])

# Mengambil tanggal referensi dari transaksi pickup paling akhir
ref_date = df_pickups['request_time'].max()

# Hitung tenure dalam hari
df_sellers['seller_tenure_days'] = (
    ref_date - df_sellers['created_at']
).dt.days

# Gabungkan kolom seller_tenure_days ke df_pickups jika diperlukan
df_pickups = df_pickups.merge(
    df_sellers[['seller_id', 'seller_tenure_days']],
    on='seller_id',
    how='left',
)

#### ⚙️ `service_name`

> 💡 Join df_pickups dengan df_services untuk mendapatkan nama layanan per transaksi.

In [ ]:
# TODO: Tambahkan service_name ke df_pickups via merge dengan df_services
df_pickups = df_pickups.merge(
    df_services[['service_code', 'service_name']],
    on='service_code',
    how='left',
)

---
### 3.2 Fitur Pilihan (Minimal 2)

Pilih minimal 2 dari: `revenue_loss_per_package`, `request_hour`, `pickup_success_rate_per_seller`, `weight_manipulation_flag`, atau fitur buatan sendiri.

#### ⚙️ Fitur Pilihan 1: Renevue Loss pe Package Calculator

**✍️ Business value dari fitur ini:**
Mengkuantifikasi potensi kerugian pendapatan akibat manipulasi berat oleh seller (selisih actual_weight vs stated_weight dikalikan tarif per kg). Membantu tim keuangan menghitung total revenue leakage yang perlu ditagih ulang.
> 

In [ ]:
# TODO: Implementasi Fitur Pilihan 1
rate_per_kg = 8000

df_pickups['weight_gap'] = (
    df_pickups['actual_weight_kg'] - df_pickups['stated_weight_kg']
).clip(lower=0)
df_pickups['revenue_loss_per_package'] = (
    df_pickups['weight_gap'] * rate_per_kg
)

#### ⚙️ Fitur Pilihan 2: Weight Manipulation Flag

**✍️ Business value dari fitur ini:**
Indikator biner untuk menyaring transaksi yang memiliki indikasi kecurangan berat secara eksplisit (misal: selisih berat > 0.5 kg atau berat klaim < 0.1 kg). Digunakan oleh tim Risk & Fraud Management untuk melakukan flagging automatis pada akun seller.
> 

In [ ]:
# TODO: Implementasi Fitur Pilihan 2
# Flag jika selisih berat > 0.5 kg atau berat klaim tidak wajar (< 0.1 kg)
df_pickups['weight_manipulation_flag'] = (
    (df_pickups['actual_weight_kg'] - df_pickups['stated_weight_kg'] > 0.5)
    | (df_pickups['stated_weight_kg'] < 0.1)
).astype(int)

---
## 4. Exploratory Data Analysis

> **Aturan:** Semua analisis menggunakan pandas — tanpa visualisasi.  
> Gunakan `.groupby()`, `.agg()`, `.value_counts()`, filtering, sorting, dan **merge antar tabel** saat dibutuhkan.  
> Setiap jawaban **wajib disertai insight** di markdown cell yang tersedia.

---
### 4.1 Analisis Berat & Revenue Leakage

**Soal 1:** Berapa rata-rata, median, dan standar deviasi `stated_weight_kg` vs `actual_volume_weight_kg`? Apa yang bisa disimpulkan dari perbedaan distribusi keduanya?

In [ ]:
# Soal 1


**✍️ Insight:**

> 

**Soal 2:** Berapa total `weight_gap_kg` keseluruhan (hanya baris dengan gap positif)? Berapa estimasi total kerugian dalam Rupiah? Dokumentasikan asumsi tarif per kg yang kamu gunakan.

In [ ]:
# Soal 2
# Dokumentasikan asumsi tarif per kg yang kamu pilih


**✍️ Insight:**

> 

**Soal 3:** Top 10 seller berdasarkan total `weight_gap_kg` kumulatif. Apakah seller-seller ini terkonsentrasi di kota tertentu atau menggunakan layanan tertentu?

In [ ]:
# Soal 3
# Hint: merge df_pickups dengan df_sellers untuk mendapatkan city per seller


**✍️ Insight:**

> 

**Soal 4:** Bandingkan rata-rata `weight_gap_kg` antara layanan **HALU** (promo murah) vs **BEST** (premium). Apakah layanan murah lebih banyak disalahgunakan untuk underdeclare berat?

In [ ]:
# Soal 4


**✍️ Insight:**

> 

---
### 4.2 Analisis SLA & Phantom Pickup

**Soal 5:** Berapa persentase `is_phantom_pickup = True` dari seluruh transaksi Success? Apakah ini tersebar merata atau terkonsentrasi pada seller atau kota tertentu?

In [ ]:
# Soal 5


**✍️ Insight:**

> 

**Soal 6:** Untuk transaksi Success yang **valid** (bukan Phantom Pickup): berapa distribusi `sla_hours`? Berapa persentase yang memenuhi SLA 1x24 jam (`is_sla_met = True`)?

In [ ]:
# Soal 6


**✍️ Insight:**

> 

**Soal 7:** Berapa distribusi `pickup_status` (Success / Failed / Rescheduled) per `service_name`? Apakah layanan tertentu lebih sering mengalami gagal pickup?

In [ ]:
# Soal 7


**✍️ Insight:**

> 

**Soal 8:** Analisis `request_hour`: pada jam berapa request pickup paling banyak terjadi? Apakah ada pola konsentrasi yang bisa digunakan untuk optimasi jadwal kurir?

In [ ]:
# Soal 8
# Hint: ekstrak jam dari request_time menggunakan .dt.hour


**✍️ Insight:**

> 

---
### 4.3 Analisis Seller & Kategori Barang

**Soal 9:** Berapa distribusi jumlah pickup per seller? Identifikasi seller 'power user' (high volume) vs seller biasa. Apakah ada perbedaan pola `weight_gap_kg` antara dua segmen ini?

In [ ]:
# Soal 9
# Tentukan sendiri threshold 'power user' dan justifikasikan


**✍️ Insight:**

> 

**Soal 10:** Berapa distribusi `is_oversize` per `item_category_clean`? Kategori barang apa yang paling sering memiliki berat aktual jauh lebih besar dari berat yang dinyatakan?

In [ ]:
# Soal 10


**✍️ Insight:**

> 

**Soal 11:** Berapa distribusi pickup per `city` (dari tabel sellers)? Kota mana yang paling banyak menghasilkan pickup request, dan kota mana yang memiliki success rate terendah?

In [ ]:
# Soal 11
# Hint: merge df_pickups dengan df_sellers untuk mendapatkan city


**✍️ Insight:**

> 

---
### 4.4 Investigasi Phantom Pickup & SLA Enforcement *(Implicit — Business Sense Required)*

> Kamu diminta **VP Operations SiCepat** untuk menyusun laporan investigasi Phantom Pickup.  
> Temuan ini akan digunakan untuk: (a) menentukan kurir mana yang perlu di-suspend,  
> (b) merancang sistem deteksi otomatis berbasis aturan (*rule-based*), dan  
> (c) mengestimasi dampak terhadap kepuasan seller.

Pilih minimal **2 angle analisis** yang paling relevan untuk menjawab kebutuhan investigasi tersebut.

#### 🔍 Investigasi — Angle 1: [Isi judul]

**✍️ Mengapa kamu memilih angle ini untuk investigasi Phantom Pickup?**

> 

In [ ]:
# Angle 1


**✍️ Insight & Rekomendasi untuk VP Operations:**

> 

#### 🔍 Investigasi — Angle 2: [Isi judul]

**✍️ Mengapa kamu memilih angle ini?**

> 

In [ ]:
# Angle 2


**✍️ Insight & Rule-Based Detection yang kamu usulkan:**

> 

---
### 4.5 Revenue Recovery & Rekomendasi Operasional *(Implicit — Open Ended)*

> Kamu diminta tim **Finance & Operations SiCepat** untuk menyusun laporan Revenue Recovery.  
> Tujuan: mengidentifikasi total kerugian yang bisa di-recover melalui *chargeback* ke platform e-commerce,  
> dan memberikan rekomendasi kebijakan untuk mencegah manipulasi dimensi di masa depan.

> 🧠 **Critical Thinking Prompt:**  
> Tidak semua `weight_gap` adalah manipulasi — beberapa mungkin error pengukuran seller yang jujur.  
> Bagaimana kamu membedakan seller yang **sengaja manipulasi** vs seller yang **tidak tahu cara mengukur dimensi**?  
> Rekomendasi terbaik mempertimbangkan trade-off: terlalu ketat bisa mengusir seller UMKM yang legitimate.

**Ekspektasi minimal:**
- Minimal 3 segmentasi berbeda untuk menganalisis revenue leakage (per service, per kategori barang, per kota seller, dll.)
- Estimasi total nilai kerugian dalam Rupiah dengan asumsi yang jelas dan terdokumentasi
- Minimal 1 rekomendasi kebijakan konkret berbasis data untuk mencegah manipulasi dimensi

**✍️ Pendekatan analisis revenue recovery yang kamu pilih:**

> 

#### 💰 Segmentasi Revenue Leakage 1: [Nama Segmentasi]

In [ ]:
# Segmentasi 1


#### 💰 Segmentasi Revenue Leakage 2: [Nama Segmentasi]

In [ ]:
# Segmentasi 2


#### 💰 Segmentasi Revenue Leakage 3: [Nama Segmentasi]

In [ ]:
# Segmentasi 3


#### 🎯 Estimasi Total Kerugian & Rekomendasi Kebijakan

In [ ]:
# Agregasi estimasi kerugian dari ketiga segmentasi
# Dokumentasikan asumsi tarif per kg yang kamu gunakan


**✍️ Rekomendasi Kebijakan untuk Mencegah Manipulasi Dimensi:**

> 

---
## 5. Export Clean Dataset

In [ ]:
# Gabungkan ketiga tabel menjadi satu dataframe final
# Gunakan LEFT JOIN dengan df_pickups sebagai tabel utama
# Sertakan semua fitur baru yang telah dibuat

# TODO: Implementasi JOIN
# df_final = df_pickups.merge(df_sellers[...], on='seller_id', how='left')
#                      .merge(df_services[...], on='service_code', how='left')

# Export
# df_final.to_csv('sicepat_clean.csv', index=False)
# print(f'Dataset berhasil disimpan: sicepat_clean.csv')
# print(f'Shape final: {df_final.shape}')
# print(f'Kolom baru yang ditambahkan: {[c for c in df_final.columns if c not in df_pickups_raw.columns]}')


---
## 6. Ringkasan & Refleksi

**Keputusan Data Cleaning yang paling challenging dan mengapa:**

> 

**Temuan paling menarik dari EDA (khususnya terkait Phantom Pickup atau revenue leakage):**

> 

**Rekomendasi bisnis utama yang bisa diberikan kepada tim Operations & Finance SiCepat:**

> 